# Curso: Introucción a Machine Learning
## Semana 9 - Parte II/II
# Bayes Ingenuo (Gaussiano): (Gaussian) Naive Bayes

Jose Alejandro Alfaro Arias

---

### Nota de esta versión documentada

Este notebook recrea el contenido del HTML entregado en clase y mantiene el mismo orden general del ejercicio.

Las secciones marcadas como **“Apunte para estudiar”** son explicaciones añadidas para facilitar el estudio. Los comentarios `# ...` dentro del código explican qué hace cada instrucción.

Cuando se hace algún ajuste práctico para poder ejecutar el notebook en VS Code, se indica expresamente.

El método de Bayes Ingenuo es válido para problemas de clasificación y hace diferencia sobre si los datos de entrada son categóricos o numéricos.

Revisa la documentación correspondiente:

- https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html
- https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html

Puedes encontrar otras variantes sobre los tipos de entradas:

- https://scikit-learn.org/stable/modules/naive_bayes.html

El caso de métodos de Bayes para regresión genera otra cantidad de modelos que no veremos por el momento.

## Apunte para estudiar: ¿qué es Naive Bayes?

**Esta explicación es añadida para estudiar; no sustituye el contenido original del profesor.**

Naive Bayes es un algoritmo de **clasificación probabilística**. Su idea central es calcular qué tan probable es que una observación pertenezca a cada clase y escoger la clase con mayor probabilidad.

El término **“ingenuo” (naive)** se debe a una suposición simplificadora: el modelo trata las variables predictoras como si fueran condicionalmente independientes entre sí una vez conocida la clase.

En este notebook se utiliza:

```python
GaussianNB()
```

La palabra **Gaussian** indica que el modelo trabaja suponiendo una distribución gaussiana o normal para las variables numéricas dentro de cada clase.

En términos prácticos, el flujo que veremos es:

1. Cargar los datos.
2. Separar variables predictoras `X` y variable objetivo `Y`.
3. Dividir los datos en entrenamiento y prueba.
4. Evaluar Gaussian Naive Bayes con validación cruzada.
5. Entrenar el modelo definitivo.
6. Revisar su matriz de confusión.
7. Compararlo con KNN mediante curvas ROC.

In [ ]:
# Librerías para manipulación de datos.
import pandas as pd
import numpy as np

# Librería para gráficos.
import matplotlib.pyplot as plt

# Counter permite contar cuántos registros existen de cada clase.
from collections import Counter

# Herramientas de Scikit-Learn para separar y validar los datos.
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

# Métrica para observar aciertos y errores por clase.
from sklearn.metrics import confusion_matrix

# Modelo Naive Bayes para variables numéricas.
from sklearn.naive_bayes import GaussianNB

## Dataset utilizado

En el HTML original se indica como referencia:

https://www.kaggle.com/datasets/kumargh/pimaindiansdiabetescsv

El ejercicio trabaja con el archivo:

```text
pima_indians_diabetes.csv
```

Para ejecutar este notebook en VS Code, coloca el CSV en la **misma carpeta** que el archivo `.ipynb`.

En esta entrega también se incluye una copia del CSV con los encabezados que utiliza el código del profesor.

## Apunte para estudiar: ¿qué queremos predecir?

El dataset contiene varias características relacionadas con pacientes y una columna final llamada `Outcome`.

En este ejercicio:

- `X` contendrá las **variables predictoras**.
- `Y` contendrá la **variable objetivo**.
- `Outcome = 0` y `Outcome = 1` representan las dos clases del problema de clasificación.

El algoritmo aprenderá patrones en las variables de entrada para intentar predecir el valor de `Outcome`.

In [ ]:
# Ruta del archivo CSV.
# Al escribir solo el nombre, Python lo buscará en la misma carpeta del notebook.
mypath = "pima_indians_diabetes.csv"

# Leemos el archivo CSV.
# sep="," indica que las columnas están separadas por comas.
# header='infer' permite que pandas utilice la primera fila como nombres de columnas.
data = pd.read_csv(mypath, sep=",", header='infer')

# Mostramos las primeras 3 filas para verificar que el archivo cargó correctamente.
data.head(3)

## Separación entre `X` y `Y`

Ahora separamos el dataset en dos partes:

- **`X`**: contiene las ocho variables que el modelo usará como información de entrada.
- **`Y`**: contiene únicamente `Outcome`, que es lo que queremos predecir.

Esta separación es una de las estructuras más comunes en Machine Learning supervisado.

In [ ]:
# Seleccionamos las columnas que funcionarán como variables predictoras.
X = data[
    [
        'Pregnancies',
        'Glucose',
        'BloodPressure',
        'SkinThickness',
        'Insulin',
        'BMI',
        'DiabetesPedigreeFunction',
        'Age'
    ]
]

# La variable objetivo es Outcome.
# Se conserva la forma utilizada en el HTML del profesor: un DataFrame de una columna.
Y = data[['Outcome']]

## Revisar la distribución de la variable objetivo

Antes de entrenar, es útil saber cuántos registros existen en cada clase.

`Counter()` cuenta cuántas veces aparece cada valor de `Outcome`.

Esto permite observar si las clases tienen cantidades similares o si una aparece con mucha más frecuencia que la otra.

In [ ]:
# Contamos cuántos registros pertenecen a cada clase.
Counter(Y['Outcome'])

## División en entrenamiento y prueba

Separamos los datos en:

- **70 % para entrenamiento**
- **30 % para prueba**

Los datos de entrenamiento se usan para construir el modelo. Los datos de prueba quedan aparte y permiten evaluar cómo se comporta el modelo con observaciones que no utilizó durante el entrenamiento.

`random_state=7` hace reproducible la división: si ejecutamos nuevamente esta celda con el mismo dataset, obtendremos la misma partición.

In [ ]:
# Dividimos X y Y simultáneamente para mantener cada observación
# asociada con su etiqueta correcta.
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    Y,
    train_size=0.7,  # 70 % de los datos se destinan a entrenamiento.
    random_state=7   # Fija la aleatoriedad para reproducir la misma división.
)

# Revisamos las dimensiones resultantes.
print("Dimensión de Xtrain:", Xtrain.shape)
print("Dimensión de Xtest :", Xtest.shape)
print("Dimensión de ytrain:", ytrain.shape)
print("Dimensión de ytest :", ytest.shape)

## Validación cruzada con `KFold`

Antes de entrenar y evaluar una única vez, el profesor utiliza **validación cruzada**.

```python
KFold(n_splits=10)
```

divide los datos de entrenamiento en **10 partes o folds**. El modelo se entrena y valida varias veces, cambiando qué fold queda temporalmente para validación.

Después:

```python
cross_val_score(...)
```

devuelve un score para cada iteración.

Finalmente usamos:

```python
sol.mean()
```

para obtener el rendimiento promedio.

### ¿Por qué aparece `np.ravel(ytrain)`?

En este notebook `ytrain` tiene forma similar a:

```text
(537, 1)
```

pero muchos estimadores de Scikit-Learn esperan la variable objetivo como un vector:

```text
(537,)
```

`np.ravel()` transforma esa estructura de una columna en un vector de una dimensión, sin cambiar los valores.

In [ ]:
# Creamos el esquema de validación cruzada.
kfold = KFold(
    n_splits=10,    # Divide el conjunto de entrenamiento en 10 folds.
    random_state=7, # Controla la aleatoriedad.
    shuffle=True    # Mezcla los datos antes de construir los folds.
)

# Creamos el modelo Gaussian Naive Bayes.
modeloGNB = GaussianNB()

# Evaluamos el modelo mediante validación cruzada.
# np.ravel(ytrain) convierte ytrain de una columna a un vector 1D.
sol = cross_val_score(
    modeloGNB,
    Xtrain,
    np.ravel(ytrain),
    cv=kfold
)

# Promedio de los scores obtenidos en los 10 folds.
print("Score promedio de validación cruzada:", sol.mean())

## Entrenamiento definitivo y matriz de confusión

Después de la validación cruzada, entrenamos `modeloGNB` utilizando todo el conjunto de entrenamiento.

Luego hacemos predicciones sobre `Xtest`.

La **matriz de confusión** permite separar los resultados en cuatro grupos:

| | Predicción 0 | Predicción 1 |
|---|---:|---:|
| **Real 0** | Verdaderos negativos | Falsos positivos |
| **Real 1** | Falsos negativos | Verdaderos positivos |

El propio HTML aclara que **los renglones representan los valores reales y las columnas las predicciones**.

In [ ]:
# Entrenamos Gaussian Naive Bayes con todos los datos de entrenamiento.
modeloGNB.fit(Xtrain, np.ravel(ytrain))

# Realizamos predicciones sobre el conjunto de prueba.
yhat = modeloGNB.predict(Xtest)

# Construimos la matriz de confusión.
# Los renglones representan las clases reales.
# Las columnas representan las clases predichas.
cm = confusion_matrix(
    ytest,
    np.ravel(yhat)
)

print("Matriz de confusión:")
print(cm)

## Comparación con K-Nearest Neighbors

El HTML introduce ahora un segundo modelo:

```python
KNeighborsClassifier()
```

La intención es contar con otro clasificador y comparar posteriormente su comportamiento con Gaussian Naive Bayes mediante una curva ROC.

Aquí KNN se utiliza con sus parámetros por defecto.

In [ ]:
# Importamos el modelo KNN.
from sklearn.neighbors import KNeighborsClassifier

# Herramienta de Scikit-Learn para construir la curva ROC.
from sklearn.metrics import RocCurveDisplay

In [ ]:
# Creamos y entrenamos un modelo KNN usando los parámetros por defecto.
modelokNN = KNeighborsClassifier().fit(
    Xtrain,
    np.ravel(ytrain)
)

# Evaluamos el modelo sobre el conjunto de prueba.
score_knn = modelokNN.score(Xtest, ytest)

print("Score de KNN en prueba:", score_knn)

## Curva ROC: Bayes frente a KNN

La última comparación del HTML utiliza una **curva ROC**.

La curva ROC muestra cómo cambia la capacidad del clasificador para detectar la clase positiva cuando cambia el umbral de decisión.

Dos conceptos importantes son:

- **TPR (True Positive Rate)**: tasa de verdaderos positivos, relacionada con el recall o sensibilidad.
- **FPR (False Positive Rate)**: tasa de falsos positivos.

En el código, primero se dibuja la curva de `modeloGNB` y luego se agrega la de `modelokNN` sobre el mismo gráfico usando:

```python
ax=gnb_curve.ax_
```

De esa forma podemos comparar visualmente ambos clasificadores en una misma figura.

> Esta explicación conceptual de ROC es un apunte añadido para estudiar; el HTML original se limita a construir el gráfico.

In [ ]:
# Dibujamos la curva ROC del modelo Gaussian Naive Bayes.
gnb_curve = RocCurveDisplay.from_estimator(
    modeloGNB,
    Xtest,
    ytest
)

# Dibujamos la curva ROC de KNN sobre los mismos ejes.
knn_curve = RocCurveDisplay.from_estimator(
    modelokNN,
    Xtest,
    ytest,
    ax=gnb_curve.ax_
)

# Se conserva la línea auxiliar utilizada en el HTML original.
plt.plot([0, 0, 1, 0], [0, 1, 1, 0], 'y--')

plt.show()

## Lectura general del ejercicio

El notebook sigue un flujo sencillo de clasificación:

1. Se carga un dataset real.
2. Se separan predictores y objetivo.
3. Se observa la distribución de clases.
4. Se crean conjuntos de entrenamiento y prueba.
5. Se evalúa Gaussian Naive Bayes mediante validación cruzada.
6. Se entrena el modelo y se revisa su matriz de confusión.
7. Se construye un KNN como modelo de comparación.
8. Se comparan ambos con curvas ROC.

### Idea clave para estudiar

No debemos quedarnos únicamente con un `score`. En este ejercicio aparecen distintas formas de observar un clasificador:

- rendimiento promedio mediante validación cruzada;
- matriz de confusión;
- score en datos de prueba;
- curva ROC.

Cada una muestra una parte distinta del comportamiento del modelo.

# Fin del Jupyter-Notebook sobre Naive Bayes - semana 9